In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: Scalars, Vectors, Matrices, Tensors ────────────────────────────────
# Scalar
lr = 0.001
print(f'Scalar: {lr}  ndim=0')

# Vector — one data point (employee)
x = np.array([28, 5, 85.0, 72000])
print(f'Vector: {x}  shape={x.shape}  ndim={x.ndim}')

# Matrix — dataset (5 employees x 4 features)
X = np.array([
    [28, 5,  85.0, 72000],
    [35, 12, 90.0, 95000],
    [42, 18, 78.0, 110000],
    [26, 2,  88.0, 58000],
    [50, 25, 72.0, 140000],
])
print(f'Matrix: shape={X.shape}  -> {X.shape[0]} samples, {X.shape[1]} features')

# Tensor — batch of images (32 images, 28x28, 1 channel)
images = np.random.rand(32, 28, 28, 1)
print(f'Tensor (batch of images): shape={images.shape}')

# Tensor — BERT embeddings (16 sentences, 128 tokens, 768 dim)
embeddings = np.random.rand(16, 128, 768)
print(f'Tensor (BERT embeddings): shape={embeddings.shape}')

In [ ]:
# ── CELL 2: Vector Operations and Dot Product ──────────────────────────────────
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print('Addition:     ', a + b)
print('Scale by 2:   ', 2 * a)
print('Normalize:    ', (a / np.linalg.norm(a)).round(4))
print('Hadamard a*b: ', a * b)
print('Dot product:  ', np.dot(a, b))

# Dot product = neuron computation
x_in = np.array([0.5, 0.8, 0.2])
w    = np.array([0.3, 0.9, 0.1])
b_n  = 0.1
neuron_out = np.dot(w, x_in) + b_n
print(f'\nNeuron: dot(w,x)+b = {neuron_out:.4f}')

# Recommendation system — dot product as preference match
user    = np.array([0.9, 0.1, 0.8])   # loves action, hates romance, loves sci-fi
movie_a = np.array([0.8, 0.2, 0.9])   # action/sci-fi film
movie_b = np.array([0.1, 0.9, 0.2])   # romance film
print(f'\nRecommender dot products:')
print(f'  user . movie_a = {np.dot(user, movie_a):.2f}  (high match)')
print(f'  user . movie_b = {np.dot(user, movie_b):.2f}  (low match)')

In [ ]:
# ── CELL 3: Matrix Multiplication — Neural Network Forward Pass ────────────────
# Shape rule: (m,k) @ (k,n) = (m,n)

A = np.array([[1,2],[3,4],[5,6]])     # (3,2)
B = np.array([[1,0,1],[0,1,0]])       # (2,3)
C = A @ B                             # (3,2)@(2,3) = (3,3)
print(f'A{A.shape} @ B{B.shape} = C{C.shape}')

# Full 2-layer neural network forward pass
np.random.seed(42)
X_data = np.array([
    [28, 5,  0.85, 0.72],
    [35, 12, 0.90, 0.95],
    [42, 18, 0.78, 1.10],
    [26, 2,  0.88, 0.58],
    [50, 25, 0.72, 1.40],
], dtype=float)

W1 = np.random.randn(4, 3) * 0.1     # (4 inputs -> 3 neurons)
b1 = np.zeros(3)
W2 = np.random.randn(3, 1) * 0.1     # (3 neurons -> 1 output)
b2 = np.zeros(1)

Z1 = X_data @ W1 + b1                # (5,4)@(4,3)+(3,) = (5,3)
A1 = np.maximum(0, Z1)               # ReLU
Z2 = A1 @ W2 + b2                    # (5,3)@(3,1)+(1,) = (5,1)

print(f'\n2-Layer Forward Pass:')
print(f'  Input X: {X_data.shape}  -> Layer1 W1: {W1.shape} -> Z1: {Z1.shape}')
print(f'  A1(relu): {A1.shape}     -> Layer2 W2: {W2.shape} -> Z2: {Z2.shape}')

# Shape error demo
try:
    _ = A @ A
except ValueError as e:
    print(f'\nShape error: {e}')
    print('  Fix: inner dims must match. (3,2)@(3,2) fails. (3,2)@(2,3) works.')

In [ ]:
# ── CELL 4: Transpose — Attention Mechanism + Normal Equations ─────────────────
A_mat = np.array([[1,2,3],[4,5,6]])  # (2,3)
print(f'A shape: {A_mat.shape}   A.T shape: {A_mat.T.shape}')
print(A_mat)
print(A_mat.T)

# Transformer attention: Q @ K.T / sqrt(d_k)
batch, seq_len, d_k = 2, 4, 8
Q = np.random.randn(batch, seq_len, d_k)
K = np.random.randn(batch, seq_len, d_k)

scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)   # (2,4,8)@(2,8,4) = (2,4,4)
print(f'\nAttention scores Q@K.T/sqrt(d):')
print(f'  Q{Q.shape} @ K.T{K.transpose(0,2,1).shape} = {scores.shape}')
print(f'  Each entry [b,i,j] = similarity of token i to token j in batch b')

# Normal equations: w = (X.T @ X)^-1 @ X.T @ y
np.random.seed(42)
n_pts, d_pts = 100, 3
X_lr = np.random.randn(n_pts, d_pts)
y_lr = X_lr @ np.array([2.0, -1.0, 0.5]) + np.random.randn(n_pts) * 0.1

w_exact = np.linalg.inv(X_lr.T @ X_lr) @ X_lr.T @ y_lr
print(f'\nLinear Regression normal equations:')
print(f'  (X.T @ X) shape: {(X_lr.T @ X_lr).shape}')
print(f'  Solved w: {w_exact.round(3)}  (true: [2.0, -1.0, 0.5])')

In [ ]:
# ── CELL 5: Vector Norms — Regularization + Cosine Similarity ─────────────────
v = np.array([3.0, -4.0, 0.0, 1.0])

l1 = np.linalg.norm(v, ord=1)
l2 = np.linalg.norm(v)
print(f'L1 norm (Manhattan):  {l1:.4f}  -> Lasso: drives weights to exactly 0')
print(f'L2 norm (Euclidean):  {l2:.4f}  -> Ridge: shrinks weights toward 0')

v_unit = v / np.linalg.norm(v)
print(f'Unit vector:          {v_unit.round(4)}  |v| = {np.linalg.norm(v_unit):.6f}')

# Cosine similarity — semantic similarity
def cosine_sim(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

king  = np.array([0.82, -0.33, 0.51, 0.12])
queen = np.array([0.79,  0.28, 0.53, 0.10])
dog   = np.array([-0.20, 0.65, -0.40, 0.55])

print(f'\nCosine similarity:')
print(f'  king vs queen: {cosine_sim(king, queen):.4f}  (similar -> high)')
print(f'  king vs dog:   {cosine_sim(king, dog):.4f}   (unrelated -> low)')

# L2 regularization penalty
weights   = np.array([2.5, -1.8, 0.3, 5.1, -0.1])
lam       = 0.01
base_loss = 0.342
l2_pen    = lam * np.sum(weights**2)
print(f'\nL2 regularization: base_loss={base_loss} + l2_penalty={l2_pen:.4f} = {base_loss+l2_pen:.4f}')
print(f'  weight=5.1 contributes most to penalty -> gradient descent shrinks it')

In [ ]:
# ── CELL 6: Linear Transformations — What Matrices Do to Data ──────────────────
np.random.seed(42)
theta  = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

transforms = {
    'Identity':  np.eye(2),
    'Scale':     np.array([[2, 0], [0, 0.5]]),
    'Rotation':  np.array([[0, -1], [1, 0]]),
    'Shear':     np.array([[1, 0.5], [0, 1]]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, T) in zip(axes, transforms.items()):
    transformed = T @ circle
    ax.plot(circle[0], circle[1], 'steelblue', alpha=0.4, label='original')
    ax.plot(transformed[0], transformed[1], 'tomato', label='transformed')
    ax.set_title(f'{name}\ndet={np.linalg.det(T):.1f}')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(True); ax.legend(fontsize=8)
plt.suptitle('Linear Transformations — What Matrices Do to Data', fontsize=12)
plt.tight_layout()
plt.show()

# Inverse — undo a transformation
A_t = np.array([[3, 1], [1, 2]], dtype=float)
x_t = np.array([5.0, 3.0])
b_t = A_t @ x_t
x_rec = np.linalg.inv(A_t) @ b_t
print(f'Original x:    {x_t}')
print(f'After A @ x:   {b_t}')
print(f'Recovered x:   {x_rec}  (via inverse)')

In [ ]:
# ── CELL 7: Eigenvalues + PCA ──────────────────────────────────────────────────
# Eigen basics
A_eig = np.array([[4, 2], [1, 3]], dtype=float)
eigvals, eigvecs = np.linalg.eig(A_eig)
print(f'Eigenvalues:  {eigvals}')
print(f'Verify Av = lambda*v: {np.allclose(A_eig @ eigvecs[:,0], eigvals[0]*eigvecs[:,0])}')

# PCA manually
np.random.seed(42)
n_pca = 200
x1 = np.random.randn(n_pca)
x2 = 0.8 * x1 + 0.2 * np.random.randn(n_pca)
X_corr = np.column_stack([x1, x2])
X_c    = X_corr - X_corr.mean(axis=0)

cov    = np.cov(X_c.T)
ev, evec = np.linalg.eigh(cov)
idx    = np.argsort(ev)[::-1]
ev, evec = ev[idx], evec[:, idx]

explained = ev / ev.sum()
print(f'\nPCA Manual: PC1 explains {explained[0]*100:.1f}% of variance')

X_pca = X_c @ evec[:, :1]
print(f'Reduced from {X_corr.shape} -> {X_pca.shape}')

# sklearn PCA on Iris (4D -> 2D)
iris   = load_iris()
pca    = PCA(n_components=2)
X_2d   = pca.fit_transform(iris.data)
print(f'\nIris PCA: {iris.data.shape} -> {X_2d.shape}')
print(f'Explained variance: {pca.explained_variance_ratio_.round(4)}')
print(f'Total: {pca.explained_variance_ratio_.sum()*100:.1f}% preserved')

fig, ax = plt.subplots(figsize=(8, 5))
for label, color, name in zip([0,1,2], ['tomato','steelblue','seagreen'], iris.target_names):
    mask = iris.target == label
    ax.scatter(X_2d[mask,0], X_2d[mask,1], c=color, label=name, alpha=0.7)
ax.set_title('Iris: PCA 4D -> 2D (classes still separable)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 8: SVD — Image Compression + Recommender System ──────────────────────
# SVD basics
A_svd = np.random.randn(4, 3)
U, s, Vt = np.linalg.svd(A_svd, full_matrices=False)
print(f'SVD: {A_svd.shape} = U{U.shape} . diag(s{s.shape}) . Vt{Vt.shape}')
print(f'Singular values: {s.round(3)}')
print(f'Reconstruction error: {np.linalg.norm(A_svd - U @ np.diag(s) @ Vt):.10f}')

# Image compression
t_img = np.linspace(0, np.pi, 100)
image = np.outer(np.sin(t_img), np.sin(2*t_img)) + 0.1 * np.random.randn(100, 100)
U_img, s_img, Vt_img = np.linalg.svd(image, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Original')
for ax, k in zip(axes[1:], [1, 5, 20]):
    approx = U_img[:,:k] @ np.diag(s_img[:k]) @ Vt_img[:k,:]
    err    = np.linalg.norm(image - approx) / np.linalg.norm(image)
    ax.imshow(approx, cmap='gray')
    ax.set_title(f'Top {k} components\nerror={err:.1%}')
for ax in axes: ax.axis('off')
plt.suptitle('SVD Image Compression', fontsize=12); plt.tight_layout(); plt.show()

# Recommender system
R = np.array([
    [5,3,0,1,0,2],[4,5,0,1,1,0],[0,0,5,4,0,3],
    [1,0,4,5,0,4],[0,1,0,0,5,4],[3,0,0,2,4,5],
], dtype=float)
U_r, s_r, Vt_r = np.linalg.svd(R, full_matrices=False)
R_pred = U_r[:,:2] @ np.diag(s_r[:2]) @ Vt_r[:2,:]
print(f'\nRecommender SVD (rank-2 approx):')
print(f'  Variance explained by top 2: {(s_r[:2]**2/np.sum(s_r**2)*100).sum():.1f}%')
print(f'  Predicted ratings row 0: {R_pred[0].round(1)}')

In [ ]:
# ── CELL 9: Broadcasting — Batch Operations ────────────────────────────────────
X_batch = np.random.randn(32, 4)
bias    = np.array([0.1, -0.2, 0.3, 0.0])

result = X_batch + bias   # (32,4) + (4,) -> broadcast bias to all 32 rows
print(f'Batch + bias: {X_batch.shape} + {bias.shape} -> {result.shape}')

# Z-score normalization via broadcasting
data = np.random.randn(1000, 5) * np.array([10, 100, 0.5, 50, 1000])
mean = data.mean(axis=0)   # (5,)
std  = data.std(axis=0)    # (5,)
norm = (data - mean) / std # (1000,5) - (5,) / (5,) -> broadcast
print(f'\nNormalization: {data.shape} -> {norm.shape}')
print(f'Mean after norm: {norm.mean(axis=0).round(8)}  (all ~0)')
print(f'Std  after norm: {norm.std(axis=0).round(4)}   (all ~1)')

# Broadcasting shapes
A_bc = np.ones((3, 4))
b_bc = np.ones((4,))
c_bc = np.ones((3, 1))
print(f'\n(3,4) + (4,)  = {(A_bc+b_bc).shape}  (b broadcast across rows)')
print(f'(3,4) + (3,1) = {(A_bc+c_bc).shape}  (c broadcast across cols)')

# Shape error
try:
    _ = np.ones((32,4)) + np.ones((3,))
except ValueError as e:
    print(f'\nBroadcast error: {e}')

In [ ]:
# ── CELL 10: Neural Network from Scratch (Pure Linear Algebra) ─────────────────
np.random.seed(42)

X_mn, y_mn = make_moons(n_samples=500, noise=0.2, random_state=42)
X_mn       = StandardScaler().fit_transform(X_mn)
y_mn       = y_mn.reshape(-1, 1).astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X_mn, y_mn, test_size=0.2)

# Weights: 2 -> 8 -> 1
W1 = np.random.randn(2, 8) * 0.1;  b1 = np.zeros((1, 8))
W2 = np.random.randn(8, 1) * 0.1;  b2 = np.zeros((1, 1))

sigmoid = lambda z: 1 / (1 + np.exp(-z))
relu    = lambda z: np.maximum(0, z)

losses = []
for epoch in range(300):
    # Forward pass — matrix multiply + broadcasting
    Z1 = X_tr @ W1 + b1;  A1 = relu(Z1)   # (n,2)@(2,8)+(1,8) = (n,8)
    Z2 = A1 @ W2 + b2;    A2 = sigmoid(Z2) # (n,8)@(8,1)+(1,1) = (n,1)
    loss = -np.mean(y_tr*np.log(A2+1e-8) + (1-y_tr)*np.log(1-A2+1e-8))
    losses.append(loss)

    # Backward pass — transpose for gradient flow
    dA2 = -(y_tr/A2 - (1-y_tr)/(1-A2)) / len(y_tr)
    dZ2 = dA2 * A2*(1-A2)
    dW2 = A1.T @ dZ2;       db2 = dZ2.sum(axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0);   dW1 = X_tr.T @ dZ1;  db1 = dZ1.sum(axis=0, keepdims=True)

    W1 -= 0.1*dW1; b1 -= 0.1*db1
    W2 -= 0.1*dW2; b2 -= 0.1*db2

Z1_t = X_te @ W1 + b1; A1_t = relu(Z1_t)
A2_t = sigmoid(A1_t @ W2 + b2)
acc  = ((A2_t > 0.5).astype(int).flatten() == y_te.flatten().astype(int)).mean()
print(f'Test accuracy after 300 epochs: {acc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(losses); axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
xx, yy = np.meshgrid(np.linspace(-3,3,100), np.linspace(-3,3,100))
grid   = np.c_[xx.ravel(), yy.ravel()]
probs  = sigmoid(relu(grid @ W1 + b1) @ W2 + b2).reshape(xx.shape)
axes[1].contourf(xx, yy, probs, levels=50, cmap='RdBu', alpha=0.7)
axes[1].scatter(X_te[:,0], X_te[:,1],
                c=y_te.flatten().astype(int), cmap='RdBu', edgecolors='k', s=20)
axes[1].set_title('Decision Boundary (learned via matrix ops)')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 11: Practice — Movie Recommender with SVD ────────────────────────────
np.random.seed(42)
movie_names = ['Action1','Action2','Romance1','Romance2','SciFi1','SciFi2']

ratings = np.array([
    [5,4,0,1,0,2],[4,5,0,1,1,0],[0,0,5,4,0,3],
    [1,0,4,5,0,4],[0,1,0,0,5,4],[3,0,0,2,4,5],
    [5,3,1,0,0,0],[0,4,0,0,1,3],[2,3,4,0,0,0],
    [0,0,3,4,5,0],
], dtype=float)

# Cosine similarity between movies
norms    = np.linalg.norm(ratings, axis=0, keepdims=True)
R_norm   = ratings / (norms + 1e-8)
sim_mat  = R_norm.T @ R_norm
print('Movie-Movie Cosine Similarity:')
for i, row in enumerate(sim_mat):
    print(f'  {movie_names[i]:10s}: ' + ' '.join(f'{v:.2f}' for v in row))

# SVD decomposition
U_r, s_r, Vt_r = np.linalg.svd(ratings, full_matrices=False)
print(f'\nSingular values: {s_r.round(2)}')
print(f'Variance by component: {(s_r**2/np.sum(s_r**2)*100).round(1)}%')

# Rank-2 approximation
k = 2
R_pred = U_r[:,:k] @ np.diag(s_r[:k]) @ Vt_r[:k,:]

# Recommend for user 1 (index 0)
user1_orig = ratings[0]
user1_pred = R_pred[0]
not_rated  = np.where(user1_orig == 0)[0]
top3       = not_rated[np.argsort(user1_pred[not_rated])[::-1][:3]]

print(f'\nUser 1 ratings: {dict(zip(movie_names, user1_orig))}')
print(f'\nTop 3 recommendations:')
for rank, idx in enumerate(top3, 1):
    print(f'  {rank}. {movie_names[idx]:10s}  (predicted: {user1_pred[idx]:.2f})')